# Toxic Comment Classification – Inference Pipeline for BERT

## 1. Imports and Labels

In [ ]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from tqdm import tqdm
import re
import os

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## 2. Clean Text

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 3. Build Dataset

In [ ]:
class JigsawTestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['comment_text'].fillna("").apply(clean_text).tolist()
        self.ids = df['id'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'id': self.ids[idx],
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

## 4. Inference

In [ ]:
def run_inference(model, dataloader, device, thresholds=None):
    model.eval()
    all_ids = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Running Inference"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.extend(probs)
            all_ids.extend(batch['id'])

    return all_ids, np.array(all_probs)

## 5. Evaluation

In [ ]:
def evaluate_predictions(ids, probs, label_csv, thresholds=None):
    labels_df = pd.read_csv(label_csv)
    preds_df = pd.DataFrame(probs, columns=LABELS)
    preds_df.insert(0, 'id', ids)

    merged = pd.merge(labels_df, preds_df, on='id', suffixes=('_true', '_pred'))
    valid_mask = (merged[[f"{label}_true" for label in LABELS]] != -1).all(axis=1)
    valid = merged[valid_mask]

    y_true = valid[[f"{label}_true" for label in LABELS]].values
    y_probs = valid[[f"{label}_pred" for label in LABELS]].values

    if thresholds is None:
        thresholds = np.full(len(LABELS), 0.5)

    y_pred = (y_probs >= thresholds).astype(int)

    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_auc = roc_auc_score(y_true, y_probs, average='macro')

    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    micro_precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
    micro_recall = recall_score(y_true, y_pred, average='micro', zero_division=0)
    micro_auc = roc_auc_score(y_true, y_probs, average='micro')

    per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_class_auc = roc_auc_score(y_true, y_probs, average=None)

    print(f"\n Evaluated on {len(valid)} samples")
    print(f"Macro F1: {macro_f1:.4f}, Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, AUC: {macro_auc:.4f}")
    print(f"Micro F1: {micro_f1:.4f}, Precision: {micro_precision:.4f}, Recall: {micro_recall:.4f}, AUC: {micro_auc:.4f}\n")

    print(" Per-class metrics:")
    for i, label in enumerate(LABELS):
        print(f"{label:15s} | F1: {per_class_f1[i]:.4f} | Precision: {per_class_precision[i]:.4f} | Recall: {per_class_recall[i]:.4f} | AUC: {per_class_auc[i]:.4f}")

    print("\n Classification Report:")
    print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

    return macro_f1

## 6. Generate Submission

In [ ]:
def generate_submission(ids, probs):
    submission_df = pd.DataFrame(probs, columns=LABELS)
    submission_df.insert(0, 'id', ids)
    submission_df = submission_df[['id'] + LABELS]
    submission_df.to_csv('submission_focal.csv', index=False)
    print("\nSubmission file 'submission_focal.csv' created successfully.")
    print("Submission DataFrame Head:")
    print(submission_df.head())

## 7. Load and Run

In [ ]:
MODEL_NAME = "bert-base-uncased"
MODEL_PATH = "best_model_bert-base-uncased_focal_full.bin"
TEST_COMMENTS_CSV = "test.csv"
TEST_LABELS_CSV = "test_labels.csv"
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
checkpoint = torch.load(MODEL_PATH, map_location=device)

model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6, problem_type="multi_label_classification")
model.load_state_dict({k.replace('model.', ''): v for k, v in checkpoint.items()})
model.to(device)

test_df = pd.read_csv(TEST_COMMENTS_CSV)[['id', 'comment_text']]
test_dataset = JigsawTestDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

thresholds = np.load("best_thresholds_bert-base-uncased_focal_full.npy", allow_pickle=True)

ids, probs = run_inference(model, test_loader, device, thresholds)

np.save("inference_ids.npy", ids)
np.save("inference_probs.npy", probs)

generate_submission(ids, probs)
f1 = evaluate_predictions(ids, probs, TEST_LABELS_CSV, thresholds)